# CMUNeXt-Plus | Polyp Segmentation (research prototype)

GitHub branch: **`research/cmunext-plus`**.

This notebook tests a *hypothesis* (context pyramid + attention-gated skip connections + deep supervision), not a proven improvement. **Never tune model choices on the 5 test datasets.**

Warm-starting your old CMUNeXt `best.pth` is faster, but because the old code selected checkpoints using CVC-ClinicDB test images and was trained on all TrainDataset, warm-start results **cannot** establish a clean paper-style comparison. For publication, re-train `cmunext` and `cmunext_plus` from scratch under identical settings.

In [ ]:
!git clone -b research/cmunext-plus https://github.com/minhhung3112/polyp-segmentation.git
%cd /content/polyp-segmentation

In [ ]:
!pip -q install -r requirements.txt

## Configure training

- Set **Runtime → Change runtime type → T4 GPU**.
- `WARM_START=False` runs from scratch and is suitable for fair baseline-vs-Plus comparison.
- To transfer your old checkpoint, upload `best.pth` to Drive, mount Drive below, and set `WARM_START=True` and `WEIGHTS` to its **actual path**.
- To continue an interrupted `cmunext_plus` run, replace `--init_weights` with `--resume` pointing at the saved Plus checkpoint.

In [ ]:
# Optional: mount Drive only if weights/checkpoint storage is needed.
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
import os, subprocess
WARM_START = False
WEIGHTS = '/content/drive/MyDrive/best.pth'  # Change this if WARM_START=True
EPOCHS = 60 if WARM_START else 100
MODEL = 'cmunext_plus'  # Change to 'cmunext' for a fair same-split baseline
SEED = 42
BATCH_SIZE = 8
LR = 5e-5 if WARM_START else 1e-4
cmd = ['python', 'train_plus.py', '--model', MODEL,
       '--epochs', str(EPOCHS), '--batchsize', str(BATCH_SIZE),
       '--lr', str(LR), '--seeds', str(SEED)]
if WARM_START:
    assert os.path.isfile(WEIGHTS), f'Checkpoint not found: {WEIGHTS}'
    cmd += ['--init_weights', WEIGHTS]
print(' '.join(cmd), flush=True)
subprocess.run(cmd, check=True)

## Save your weights

Colab files are temporary. Save `snapshots/<model>/best.pth` to your own Drive folder before the session closes.

In [ ]:
import pathlib
best = pathlib.Path('snapshots') / MODEL / 'best.pth'
print('BEST CHECKPOINT:', best, 'FOUND:', best.exists())
# If Drive mounted, uncomment and adjust target:
# import shutil
# shutil.copy2(best, '/content/drive/MyDrive/cmunext_plus_best.pth')

## Final test (only after model/seed/loss selection has been frozen)

The five test datasets (CVC-300, CVC-ClinicDB, CVC-ColonDB, ETIS, Kvasir) must **not** guide repeated model selection. See [`docs/CMUNEXT_PLUS.md`](https://github.com/minhhung3112/polyp-segmentation/blob/research/cmunext-plus/docs/CMUNEXT_PLUS.md) for protocol and caveats.

In [ ]:
# Run ONCE on the chosen final checkpoint:
# !python test.py --model cmunext_plus --weights snapshots/cmunext_plus/best.pth